# DL Masterclass 01: Neural Network Mathematics & Architecture

Welcome to the first notebook of the **Deep Learning (Theory & Practice) Sequence**.

This track focuses purely on the underlying mechanics of dense neural networks (Multi-Layer Perceptrons). We strip away the PyTorch abstractions to understand the exact calculus of backpropagation, the realities of high-dimensional optimization, and how these models physically execute on GPU silicon.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin. It addresses the fundamental architecture of modern AI:

> *"The Universal Approximation Theorem mathematically guarantees that a single hidden layer can approximate ANY continuous function. Why, then, in practice, do we always build networks with 100 deep layers instead of 1 incredibly wide shallow layer?"*

---

## Table of Contents
1. **Universal Approximation vs. Deep Efficiency:** Why depth matters more than width.
2. **The Forward Pass:** Matrix dimensions and non-linearities.
3. **The Backward Pass (Derivation):** Unpacking the Chain Rule and the Transposed Weight Matrix.
4. **Scratch Implementation:** Building a 3-layer MLP using pure NumPy arrays.


## 1. Universal Approximation vs. Deep Efficiency

The **Universal Approximation Theorem** states that a neural network with just one hidden layer containing a finite number of neurons can approximate any continuous function to any desired degree of accuracy.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *If 1 layer is mathematically enough, why use 100 layers?*

**A:** **Sample Efficiency and Feature Hierarchies.**

To approximate a highly complex function (like distinguishing a dog from a cat) using a single layer, the number of required neurons grows exponentially. You might need $10^{50}$ neurons in that single layer, which requires more RAM than exists in the universe and an infinite amount of training data to prevent overfitting.

Deep networks solve this by learning **hierarchical compositions**. 
- Layer 1 learns edges.
- Layer 2 combines edges into curves.
- Layer 3 combines curves into faces.

Mathematically, a deep network can express the exact same function as a wide shallow network, but it requires exponentially *fewer* neurons and exponentially *less* training data to do it. Depth forces the network to discover reusable, modular abstractions.

## 2. The Forward Pass (Matrix Architecture)

A standard Dense (Linear) layer is mathematically defined as:
$Z^{(l)} = X \cdot W^{(l)} + b^{(l)}$
$A^{(l)} = \sigma(Z^{(l)})$

Where:
*   $X$: The input batch (Shape: `[batch_size, input_features]`)
*   $W^{(l)}$: The weight matrix (Shape: `[input_features, output_neurons]`)
*   $b^{(l)}$: The bias vector (Shape: `[output_neurons]`)
*   $Z^{(l)}$: The pre-activation linear sum (Shape: `[batch_size, output_neurons]`)
*   $\sigma$: The non-linear activation function (e.g., ReLU, Sigmoid).
*   $A^{(l)}$: The final activated output passed to the next layer.

If we remove the non-linear activation $\sigma$, a 100-layer neural network collapses into a single linear regression model. Matrix multiplication is associative: $(X \cdot W_1) \cdot W_2 = X \cdot (W_1 \cdot W_2)$. Without non-linearities, deep learning does not exist.

## 3. The Backward Pass (Pure Calculus)

To update a specific weight matrix $W^{(l)}$, we need the gradient of the Loss function $L$ with respect to that weight: $\frac{\partial L}{\partial W^{(l)}}$.

We use the **Chain Rule of Calculus** to propagate the error backward layer by layer.

Let $\delta^{(l)} = \frac{\partial L}{\partial Z^{(l)}}$ (The error signal arriving at layer $l$).

1.  **Gradient for Weights:** How did $W$ contribute to the error at $Z$?
    $\frac{\partial L}{\partial W^{(l)}} = A^{(l-1)T} \cdot \delta^{(l)}$
    *(Note: We cross-multiply the activations of the previous layer with the incoming error).* 

2.  **Gradient for Biases:**
    $\frac{\partial L}{\partial b^{(l)}} = \sum \delta^{(l)}$ (summed across the batch dimension).

3.  **Passing the Error Backwards:** How do we calculate $\delta^{(l-1)}$ for the previous layer to use?
    $\delta^{(l-1)} = (\delta^{(l)} \cdot W^{(l)T}) \odot \sigma'(Z^{(l-1)})$
    *(Note: $\odot$ is element-wise multiplication. $\sigma'$ is the derivative of the activation function).*

### 🎓 DEEP QUESTION 2 ANSWERED
**Q:** *During the backward pass (Step 3), why do we multiply the error by the Transpose of the weight matrix ($W^T$)?*

**A:** During the Forward Pass, $W$ routes information from $N$ inputs to $M$ outputs. If Input node $X_1$ connects strongly to Output node $Z_5$ via weight $w_{1,5}$, then during the Backward Pass, it is mathematically required that the Error at node $Z_5$ flows strongly back into $X_1$ via that exact same channel $w_{1,5}$. 

Transposing the matrix $W^T$ mechanically reverses the routing framework. It takes the $M$ incoming errors and splays them backward across the $N$ input nodes proportionally to how the inputs originally created the outputs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

# -----------------------------------------------------
# IMPLEMENTATION: 3-Layer MLP Backprop from Scratch
# -----------------------------------------------------

# 1. Create a non-linear dataset (Moons)
X, y = make_moons(n_samples=1000, noise=0.1, random_state=42)
y = y.reshape(-1, 1) # Shape: (1000, 1)

# 2. Define Network Architecture
input_dim = 2
hidden_dim = 16
output_dim = 1

# Initialize Weights (Xavier/Glorot style)
np.random.seed(42)
W1 = np.random.randn(input_dim, hidden_dim) / np.sqrt(input_dim)
b1 = np.zeros((1, hidden_dim))
W2 = np.random.randn(hidden_dim, output_dim) / np.sqrt(hidden_dim)
b2 = np.zeros((1, output_dim))

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# 3. Training Loop
learning_rate = 0.1
epochs = 2000
losses = []

for epoch in range(epochs):
    # --- FORWARD PASS ---
    # Layer 1
    Z1 = X.dot(W1) + b1
    A1 = relu(Z1)
    
    # Layer 2 (Output)
    Z2 = A1.dot(W2) + b2
    A2 = sigmoid(Z2) # Probabilities
    
    # Calculate Log-Loss (Binary Cross Entropy)
    # adding tiny epsilon to prevent log(0) errors
    loss = -np.mean(y * np.log(A2 + 1e-15) + (1 - y) * np.log(1 - A2 + 1e-15))
    losses.append(loss)
    
    # --- BACKWARD PASS (The Calculus) ---
    # Note: The derivative of Log-Loss combined with the Sigmoid activation 
    # miraculously simplifies to exactly (Prediction - True Label)
    dZ2 = A2 - y  # Error signal at the exact output node. Shape: (Batch, 1)
    
    # Gradients for Layer 2
    dW2 = A1.T.dot(dZ2) / len(X)
    db2 = np.sum(dZ2, axis=0, keepdims=True) / len(X)
    
    # Propagate Error backwards to Layer 1 using the Transpose of W2
    # Then multiply by the derivative of the local activation function (ReLU)
    dA1 = dZ2.dot(W2.T)
    dZ1 = dA1 * relu_derivative(Z1)
    
    # Gradients for Layer 1
    dW1 = X.T.dot(dZ1) / len(X)
    db1 = np.sum(dZ1, axis=0, keepdims=True) / len(X)

    # --- OPTIMIZATION STEP ---
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

    if epoch % 500 == 0:
        print(f"Epoch {epoch}: Loss = {loss:.4f}")

# 4. Plot Convergence
plt.figure(figsize=(8,4))
plt.plot(losses, color='purple', lw=2)
plt.title("Numpy Manual Backpropagation: Loss Convergence")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.show()

# You have just written PyTorch.
